# 🫀 Project 1 — Heart Disease Risk Prediction

**UCI dataset ID:** 45  
**Project type:** Binary classification  
**Goal:** Predict whether heart disease is present from patient-related measurements.

> **Educational/research project only — not a clinical diagnostic system.**

## Executive summary

This project demonstrates a complete beginner-to-intermediate machine-learning workflow in Python:

- load and inspect a UCI dataset
- clean missing values
- convert the original multi-level disease target into a binary target
- explore the data with visualizations
- split data into training and test sets
- build Logistic Regression, SVM and Random Forest models
- compare models with stratified cross-validation
- evaluate the selected model on unseen test data
- interpret model behavior with coefficients or feature importance
- explain limitations and practical implications

### Why this project matters

In a classification problem, a model can make two important types of mistakes:

- **False negative:** disease is present but the model predicts no disease.
- **False positive:** disease is absent but the model predicts disease.

For this educational project, **recall is emphasized** because missing a positive case is considered an important error to monitor.


## 📚 How to read this notebook

Each major section has a short explanation **outside the code cell**.

The learning pattern is:

**Question → Data → Cleaning → EDA → Split → Model → Validation → Test → Interpretation → Conclusion**

Run the notebook from top to bottom so that every variable is created before it is used.


### 🧰 Step 1 — Set up the Python environment

We install the UCI dataset helper and import the main Python libraries used for data science, visualization and machine learning.


In [ ]:
!pip -q install ucimlrepo

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Environment ready.")


### 📥 Step 2 — Load the UCI Heart Disease dataset

The `ucimlrepo` package retrieves the dataset directly from the UCI Machine Learning Repository. We keep the feature table and target separate so the prediction problem is explicit.


In [ ]:
from ucimlrepo import fetch_ucirepo

heart = fetch_ucirepo(id=45)

X_raw = heart.data.features.copy()
y_raw = heart.data.targets.copy()

print("Feature shape:", X_raw.shape)
print("Target shape:", y_raw.shape)

display(X_raw.head())
display(y_raw.head())


### 🔎 Step 3 — Inspect the raw data

Before changing anything, inspect column names, data types, missing values and target values. This is an important data-science habit: **understand the raw data before cleaning it**.


In [ ]:
print("Columns:")
print(list(X_raw.columns))

print("\nData types:")
display(X_raw.dtypes.to_frame("dtype"))

print("\nRaw missing-value markers:")
display(X_raw.isin(["?", "NA", "N/A", ""]).sum().sort_values(ascending=False).to_frame("count"))

print("\nRaw target values:")
display(y_raw.iloc[:, 0].value_counts(dropna=False).sort_index())


### 🧹 Step 4 — Preserve raw data and clean the predictors

We create working copies so the downloaded data remains untouched. Common missing-value markers are converted to `NaN`, and predictor columns are converted to numeric values where possible.

The actual imputation is deliberately postponed until after the train/test split. This prevents information from the test set leaking into the training process.


In [ ]:
raw_X = X_raw.copy()
raw_y = y_raw.copy()

X = X_raw.replace(["?", "NA", "N/A", ""], np.nan).copy()
X = X.apply(pd.to_numeric, errors="coerce")

print("Missing values after standardization:")
display(X.isna().sum().sort_values(ascending=False).to_frame("missing"))


### 🎯 Step 5 — Define the binary target

The UCI Heart Disease target can represent disease severity. For this project, the target is simplified into:

- `0` → no disease
- `1` → disease present

This converts the problem into a **binary classification** task.


In [ ]:
target_col = y_raw.columns[0]
y_numeric = pd.to_numeric(y_raw[target_col], errors="coerce")

valid_rows = y_numeric.notna()
X = X.loc[valid_rows].copy()
y_numeric = y_numeric.loc[valid_rows].copy()

y = (y_numeric > 0).astype(int)

print("Binary target distribution:")
display(
    y.value_counts()
     .sort_index()
     .rename(index={0: "No disease", 1: "Disease"})
     .to_frame("count")
)


### 📊 Step 6 — Exploratory Data Analysis

EDA helps us understand the data before modeling. We inspect class balance, numerical distributions and relationships between variables.

The goal is not to prove causation. The goal is to discover patterns that may be useful for modeling and to identify potential data-quality issues.


In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(x=y, order=[0, 1])
plt.title("Heart Disease Class Distribution")
plt.xlabel("Target (0 = No disease, 1 = Disease)")
plt.ylabel("Count")
plt.show()

numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

display(X[numeric_cols].describe().T)

plt.figure(figsize=(11, 7))
corr_df = pd.concat([X[numeric_cols], y.rename("target")], axis=1)
sns.heatmap(corr_df.corr(), center=0)
plt.title("Correlation Heatmap")
plt.show()


### 📈 Step 7 — Compare selected variables by target

Boxplots provide a simple way to compare the distribution of important numerical variables between the two target groups.

A visible difference does **not** mean that the variable causes disease. It only shows that the distributions differ in this dataset.


In [ ]:
candidate_cols = [c for c in ["age", "trestbps", "chol", "thalach"] if c in X.columns]

for col in candidate_cols:
    plt.figure(figsize=(7, 4))
    sns.boxplot(x=y, y=X[col])
    plt.title(f"{col} by Disease Status")
    plt.xlabel("Disease (0 = No, 1 = Yes)")
    plt.ylabel(col)
    plt.show()


### ✂️ Step 8 — Create the train/test split

We keep a portion of the data completely unseen until final evaluation.

`stratify=y` preserves the approximate class balance in both sets. The test set is not used to choose the model.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("\nTraining class distribution:")
display(y_train.value_counts(normalize=True).sort_index().rename("proportion").to_frame())


### 🧱 Step 9 — Build leakage-safe preprocessing and models

A `Pipeline` keeps preprocessing and modeling together.

- **Median imputation** fills missing numerical values using training data.
- **StandardScaler** puts numerical variables on comparable scales for Logistic Regression and SVM.
- **Random Forest** does not require feature scaling, but it still uses the same safe imputation approach.

Keeping preprocessing inside the pipeline prevents accidental leakage during cross-validation.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),
    "SVM": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC(
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=400,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
}

print("Models prepared:", list(models.keys()))


### 🔁 Step 10 — Compare models with stratified cross-validation

Cross-validation gives a more reliable estimate of model performance than relying on a single training split.

We calculate:

- **Accuracy** — overall proportion of correct predictions
- **Precision** — proportion of predicted positives that are actually positive
- **Recall** — proportion of actual positives that are detected
- **F1** — balance between precision and recall
- **ROC-AUC** — ranking/discrimination performance across thresholds

For this project, recall is the primary selection metric.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_rows = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_rows.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean()
    })

cv_results = (
    pd.DataFrame(cv_rows)
    .sort_values(["Recall", "F1", "ROC-AUC"], ascending=False)
    .reset_index(drop=True)
)

display(cv_results.style.format({
    "Accuracy": "{:.3f}",
    "Precision": "{:.3f}",
    "Recall": "{:.3f}",
    "F1": "{:.3f}",
    "ROC-AUC": "{:.3f}"
}))


### 🏆 Step 11 — Select the model

The supplied project proposal emphasizes recall. We therefore select the model with the strongest cross-validated recall, using F1 and ROC-AUC as tie-breakers.

This is a **model-selection decision**, not a claim that one algorithm is universally best.


In [ ]:
best_name = cv_results.iloc[0]["Model"]
best_model = models[best_name]

print("Selected model:", best_name)
print("Selection priority: Recall → F1 → ROC-AUC")


### 🧪 Step 12 — Train the selected model and evaluate unseen test data

Now the selected pipeline is fitted only on the training set and evaluated once on the untouched test set.

This gives us a more honest estimate of how the model performs on data it did not see during training or model selection.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

test_metrics = pd.DataFrame([{
    "Model": best_name,
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}])

display(test_metrics.style.format({
    "Accuracy": "{:.3f}",
    "Precision": "{:.3f}",
    "Recall": "{:.3f}",
    "F1": "{:.3f}",
    "ROC-AUC": "{:.3f}"
}))

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["No disease", "Disease"]
))


### 🔍 Step 13 — Inspect the confusion matrix

The confusion matrix breaks predictions into:

- **True Negative (TN):** correctly predicted no disease
- **False Positive (FP):** predicted disease when there was no disease
- **False Negative (FN):** missed a disease case
- **True Positive (TP):** correctly detected disease

This is especially useful when recall is important.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No disease", "Disease"]
)
disp.plot()
plt.title(f"Confusion Matrix — {best_name}")
plt.show()


### 📉 Step 14 — Inspect the ROC curve

The ROC curve shows the trade-off between the true-positive rate and false-positive rate across classification thresholds.

A ROC-AUC closer to 1 indicates stronger discrimination, while a value around 0.5 indicates performance close to random ranking.


In [ ]:
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title(f"ROC Curve — {best_name}")
plt.show()


### 🔎 Step 15 — Interpret the selected model

Interpretation depends on the model type.

- **Logistic Regression:** coefficients show the direction and relative strength of standardized predictors.
- **Random Forest:** feature importance measures how useful features were for reducing impurity across the trees.
- **SVM:** direct feature importance is not available in the same simple form.

These are associations used for model interpretation, **not medical causal conclusions**.


In [ ]:
if best_name == "Logistic Regression":
    model_step = best_model.named_steps["model"]
    coefficients = pd.Series(
        model_step.coef_[0],
        index=X_train.columns
    ).sort_values()

    print("Most negative coefficients:")
    display(coefficients.head(10).to_frame("coefficient"))

    print("Most positive coefficients:")
    display(coefficients.tail(10).sort_values(ascending=False).to_frame("coefficient"))

    plt.figure(figsize=(8, 6))
    coefficients.tail(10).sort_values().plot(kind="barh")
    plt.title("Top Positive Logistic Regression Coefficients")
    plt.xlabel("Coefficient")
    plt.show()

elif best_name == "Random Forest":
    model_step = best_model.named_steps["model"]
    importances = pd.Series(
        model_step.feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=False)

    display(importances.head(15).to_frame("importance"))

    plt.figure(figsize=(8, 6))
    importances.head(15).sort_values().plot(kind="barh")
    plt.title("Top Random Forest Feature Importances")
    plt.xlabel("Importance")
    plt.show()

else:
    print("SVM selected: direct feature importance is not reported here.")


### 🧠 Step 16 — Translate the results into data-science language

Use the outputs above to answer four questions:

1. **What model performed best under the chosen selection rule?**
2. **How well did it perform on unseen data?**
3. **How many positive cases were missed?**
4. **Which variables were most influential for the selected model?**

A strong portfolio project does not stop at a score. It explains what the score means and what the model cannot tell us.


In [ ]:
print("PROJECT RESULT SUMMARY")
print("=" * 60)
print(f"Selected model: {best_name}")
print(f"Test accuracy: {test_metrics.loc[0, 'Accuracy']:.3f}")
print(f"Test precision: {test_metrics.loc[0, 'Precision']:.3f}")
print(f"Test recall: {test_metrics.loc[0, 'Recall']:.3f}")
print(f"Test F1: {test_metrics.loc[0, 'F1']:.3f}")
print(f"Test ROC-AUC: {test_metrics.loc[0, 'ROC-AUC']:.3f}")
print(f"Test false negatives: {cm[1, 0]}")


## ⚠️ Limitations

This project should be interpreted as an educational machine-learning exercise.

Important limitations include:

- the dataset is relatively small compared with modern clinical datasets
- the observations are not a substitute for clinical validation
- the data is observational, so model associations should not be interpreted as causation
- performance estimates can vary with the train/test split
- the model has not been externally validated on a separate population
- model thresholds and costs should be chosen according to the real-world application
- this notebook does not provide medical diagnosis or treatment advice


## 💼 Portfolio takeaway

This project demonstrates practical skills in:

**Python → pandas → NumPy → EDA → data cleaning → train/test splitting → preprocessing pipelines → classification → cross-validation → model selection → evaluation → interpretation → communication**

The most important lesson is not memorizing one algorithm. It is learning to move systematically from a real-world question to a reproducible, evaluated and clearly explained machine-learning solution.


## 💾 Step 17 — Save reusable outputs

The final section saves the cleaned dataset, cross-validation comparison and test metrics so the analysis can be reused in a report or portfolio.


In [ ]:
os.makedirs("project1_outputs", exist_ok=True)

X.to_csv("project1_outputs/heart_features_clean.csv", index=False)
pd.DataFrame({"target": y}).to_csv("project1_outputs/heart_target.csv", index=False)
cv_results.to_csv("project1_outputs/model_comparison_cv.csv", index=False)
test_metrics.to_csv("project1_outputs/test_metrics.csv", index=False)

print("Saved outputs to project1_outputs/")


# ✅ Completion checklist

- [x] Define the prediction problem
- [x] Preserve raw data before cleaning
- [x] Convert missing markers to `NaN`
- [x] Binarize the original disease target
- [x] Perform exploratory data analysis
- [x] Use a stratified train/test split
- [x] Use preprocessing pipelines
- [x] Compare Logistic Regression, SVM and Random Forest
- [x] Use stratified cross-validation
- [x] Prioritize recall during model selection
- [x] Evaluate the selected model on unseen test data
- [x] Inspect the confusion matrix and ROC curve
- [x] Interpret feature influence
- [x] State limitations and avoid causal/clinical claims
- [x] Save reusable project outputs

**Next project:** Obesity Levels Prediction
